
# Task 2: Machine Learning Data Preparation
## Credit Card Fraud Detection

**Objective:** Prepare the cleaned dataset for machine learning through feature engineering, data splitting, preprocessing, and validation, without training any machine learning model.
  


### 1.1 Import Libraries and Load Dataset

Import the required Python libraries and load the training and testing datasets prepared in Task 1.

Display the shape of each dataset to verify that the files were loaded successfully.


In [13]:
import pandas as pd
import numpy as np

df_train = pd.read_csv("archive/fraudTrain.csv")
df_test = pd.read_csv("archive/fraudTest.csv")

print("Train Shape:", df_train.shape)
print("Test Shape:", df_test.shape)

Train Shape: (1296675, 23)
Test Shape: (555719, 23)



### 1.2 Identify Features

Examine the dataset columns and their data types to identify numerical, categorical, and date-related features.

Analyze the target variable (`is_fraud`) to understand the distribution of fraudulent and legitimate transactions.


In [14]:
# Display column names and data types
print("Dataset Features:")
print(df_train.dtypes)

# Check target distribution
print("\nTarget Distribution:")
print(df_train["is_fraud"].value_counts())

# Display first 5 rows
display(df_train.head())

Dataset Features:
Unnamed: 0                 int64
trans_date_trans_time     object
cc_num                     int64
merchant                  object
category                  object
amt                      float64
first                     object
last                      object
gender                    object
street                    object
city                      object
state                     object
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                       object
dob                       object
trans_num                 object
unix_time                  int64
merch_lat                float64
merch_long               float64
is_fraud                   int64
dtype: object

Target Distribution:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0



### 1.3 Create New Features

Create additional features to improve the representation of transaction behavior:

- **trans_hour:** Hour of the transaction.
- **trans_day:** Day of the week.
- **customer_age:** Customer's age at the transaction date.
- **distance_km:** Distance between the customer and merchant.

These features are generated without using the target variable to avoid target leakage.


In [15]:
def create_features(df):
    df = df.copy()

    # Convert dates
    df["trans_date_trans_time"] = pd.to_datetime(
        df["trans_date_trans_time"]
    )
    df["dob"] = pd.to_datetime(df["dob"])

    # Transaction time features
    df["trans_hour"] = df["trans_date_trans_time"].dt.hour
    df["trans_day"] = df["trans_date_trans_time"].dt.dayofweek

    # Customer age at transaction time
    df["customer_age"] = (
        df["trans_date_trans_time"].dt.year
        - df["dob"].dt.year
        - (
            (df["trans_date_trans_time"].dt.month * 100
             + df["trans_date_trans_time"].dt.day)
            <
            (df["dob"].dt.month * 100 + df["dob"].dt.day)
        )
    )

    # Distance between customer and merchant
    lat1 = np.radians(df["lat"])
    lon1 = np.radians(df["long"])
    lat2 = np.radians(df["merch_lat"])
    lon2 = np.radians(df["merch_long"])

    a = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin((lon2 - lon1) / 2) ** 2
    )

    a = np.clip(a, 0, 1)

    df["distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))

    return df


df_train = create_features(df_train)
df_test = create_features(df_test)

print("Train Shape:", df_train.shape)
print("Test Shape:", df_test.shape)

display(
    df_train[
        ["trans_hour", "trans_day", "customer_age", "distance_km"]
    ].head()
)

Train Shape: (1296675, 27)
Test Shape: (555719, 27)


,trans_hour,trans_day,customer_age,distance_km
0,0,1,30,78.597568
1,0,1,40,30.212176
2,0,1,56,108.206083
3,0,1,51,95.673231
4,0,1,32,77.556744



### 1.4 Remove Unnecessary Features

Remove identifiers, personal information, and redundant columns that are not needed for model preparation.

Keep relevant numerical and categorical features for subsequent preprocessing.


In [16]:
# Columns to remove
drop_columns = [
    "Unnamed: 0",
    "cc_num",
    "trans_num",
    "first",
    "last",
    "street",
    "trans_date_trans_time",
    "dob",
    "unix_time"
]

# Remove unnecessary columns
df_train = df_train.drop(columns=drop_columns)
df_test = df_test.drop(columns=drop_columns)

# Check remaining features
print("Train Shape:", df_train.shape)
print("Test Shape:", df_test.shape)

print("\nRemaining Features:")
print(df_train.columns.tolist())

Train Shape: (1296675, 18)
Test Shape: (555719, 18)

Remaining Features:
['merchant', 'category', 'amt', 'gender', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'merch_lat', 'merch_long', 'is_fraud', 'trans_hour', 'trans_day', 'customer_age', 'distance_km']



### 1.5 Review Feature Engineering Changes

Review and document the feature engineering changes.

- **Added:** Transaction hour, day of the week, customer age, and customer-to-merchant distance.
- **Removed:** Identifiers, personal information, and redundant date/time columns.
- **Retained:** Relevant numerical and categorical transaction features.

All new features were created without using the target variable to prevent target leakage.


In [26]:

# Feature engineering summary
feature_review = pd.DataFrame({
    "Feature": [
        "trans_hour",
        "trans_day",
        "customer_age",
        "distance_km",
        "cc_num",
        "trans_num",
        "first",
        "last",
        "street",
        "trans_date_trans_time",
        "dob",
        "unix_time",
        "Unnamed: 0"
    ],
    "Action": ["Added"] * 4 + ["Removed"] * 9,
    "Reason": [
        "Transaction hour",
        "Day of the week",
        "Customer age at transaction",
        "Distance between customer and merchant",
        "Card identifier",
        "Transaction identifier",
        "Personal information",
        "Personal information",
        "Personal information",
        "Replaced by engineered time features",
        "Replaced by customer_age",
        "Redundant timestamp",
        "Row index"
    ]
})

display(feature_review)

feature_review.to_csv(
    "task2_output/feature_review.csv",
    index=False
)


,Feature,Action,Reason
0,trans_hour,Added,Transaction hour
1,trans_day,Added,Day of the week
2,customer_age,Added,Customer age at transaction
3,distance_km,Added,Distance between customer and merchant
4,cc_num,Removed,Card identifier
5,trans_num,Removed,Transaction identifier
6,first,Removed,Personal information
7,last,Removed,Personal information
8,street,Removed,Personal information
9,trans_date_trans_time,Removed,Replaced by engineered time features



## 02 Data Splitting

### 2.1 Define Input Features (X) and Target (y)

Separate the input features from the target variable (`is_fraud`).

The dataset already provides separate training and testing files, so the original split is preserved for evaluation.

No machine learning model will be selected or trained in this task.


In [17]:

# Define features and target
X_train = df_train.drop(columns=["is_fraud"])
y_train = df_train["is_fraud"]

X_test = df_test.drop(columns=["is_fraud"])
y_test = df_test["is_fraud"]

# Check shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

# Verify matching feature columns
print(
    "\nSame feature columns:",
    X_train.columns.equals(X_test.columns)
)


X_train: (1296675, 17)
X_test: (555719, 17)
y_train: (1296675,)
y_test: (555719,)

Same feature columns: True



## 03 Data Preprocessing

### 3.1 Handle Missing Values

Check the training and testing datasets for missing values.

Use suitable imputation techniques when needed:
- Numerical features: Median imputation.
- Categorical features: Most frequent value imputation.

Preprocessing will be fitted on the training data only to prevent data leakage.


In [18]:

# Check missing values
print("Train Missing Values:")
print(X_train.isnull().sum())

print("\nTest Missing Values:")
print(X_test.isnull().sum())


Train Missing Values:
merchant        0
category        0
amt             0
gender          0
city            0
state           0
zip             0
lat             0
long            0
city_pop        0
job             0
merch_lat       0
merch_long      0
trans_hour      0
trans_day       0
customer_age    0
distance_km     0
dtype: int64

Test Missing Values:
merchant        0
category        0
amt             0
gender          0
city            0
state           0
zip             0
lat             0
long            0
city_pop        0
job             0
merch_lat       0
merch_long      0
trans_hour      0
trans_day       0
customer_age    0
distance_km     0
dtype: int64



### 3.2 Identify Numerical and Categorical Features

Separate the input features into numerical and categorical columns.

Numerical features will be imputed and scaled, while categorical features will be imputed and encoded.

This ensures that all features can be converted into a suitable numerical format for machine learning.


In [19]:

# Identify feature types
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

print("\nTotal Numerical:", len(numerical_features))
print("Total Categorical:", len(categorical_features))


Numerical Features:
['amt', 'zip', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'distance_km']

Categorical Features:
['merchant', 'category', 'gender', 'city', 'state', 'job']

Total Numerical: 8
Total Categorical: 6



### 3.3 Verify Engineered Features

Verify that all engineered features are present in the training and testing datasets before applying preprocessing.

This ensures that the newly created features are included in the machine learning inputs.


In [20]:

expected_features = [
    "trans_hour",
    "trans_day",
    "customer_age",
    "distance_km"
]

print("Missing from X_train:")
print([col for col in expected_features if col not in X_train.columns])

print("\nMissing from X_test:")
print([col for col in expected_features if col not in X_test.columns])

print("\nAll X_train columns:")
print(X_train.columns.tolist())


Missing from X_train:
[]

Missing from X_test:
[]

All X_train columns:
['merchant', 'category', 'amt', 'gender', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'merch_lat', 'merch_long', 'trans_hour', 'trans_day', 'customer_age', 'distance_km']



### 3.4 Validate and Fix Feature Types

Ensure that all engineered features have appropriate numerical data types.

Reidentify numerical and categorical columns to include all 17 input features before preprocessing.


In [21]:

# Convert engineered features to numerical types
engineered_features = [
    "trans_hour",
    "trans_day",
    "customer_age",
    "distance_km"
]

for col in engineered_features:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

# Reidentify feature types
numerical_features = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical Features:", len(numerical_features))
print("Categorical Features:", len(categorical_features))

print("\nTotal Features:",
      len(numerical_features) + len(categorical_features))

print("\nUnclassified Features:",
      X_train.columns.difference(
          numerical_features + categorical_features
      ).tolist())


Numerical Features: 11
Categorical Features: 6

Total Features: 17

Unclassified Features: []



### 3.5 Build Preprocessing Pipeline

Create preprocessing pipelines for numerical and categorical features.

- **Numerical Pipeline:** Apply median imputation and standardization.
- **Categorical Pipeline:** Apply most-frequent imputation and one-hot encoding.
- **ColumnTransformer:** Combine both pipelines into a single preprocessing workflow.

The preprocessing steps will be fitted only on the training data to prevent data leakage.


In [22]:

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing Pipeline Created Successfully!")


Preprocessing Pipeline Created Successfully!



### 3.6 Apply Preprocessing to Train and Test Data

Fit the preprocessing pipeline on the training dataset and apply the same transformations to the testing dataset.

- Use `fit_transform()` on training features.
- Use `transform()` on testing features.
- Maintain consistent feature transformations across both datasets.
- Prevent data leakage by learning preprocessing parameters from training data only.

No machine learning model is trained during this step.


In [23]:

# Fit preprocessing on training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same preprocessing to test data
X_test_processed = preprocessor.transform(X_test)

# Display processed data shapes
print("Processed Train Shape:", X_train_processed.shape)
print("Processed Test Shape:", X_test_processed.shape)

print("\nPreprocessing Applied Successfully!")


Processed Train Shape: (1296675, 2159)
Processed Test Shape: (555719, 2159)

Preprocessing Applied Successfully!



## 04 Final Data Checks

### 4.1 Validate Processed Data

Verify that the processed training and testing datasets are ready for machine learning.

- Check the final data shapes.
- Confirm that both datasets have the same number of features.
- Check for missing and infinite values.
- Display a sample of the processed features.


In [24]:

import numpy as np
from scipy import sparse

# Check shapes
print("Train Shape:", X_train_processed.shape)
print("Test Shape:", X_test_processed.shape)

# Check feature consistency
print(
    "\nSame Number of Features:",
    X_train_processed.shape[1] == X_test_processed.shape[1]
)

# Check for missing or infinite values
def check_invalid_values(data):
    values = data.data if sparse.issparse(data) else data
    return {
        "Missing Values": np.isnan(values).sum(),
        "Infinite Values": np.isinf(values).sum()
    }

print("\nTrain Validation:")
print(check_invalid_values(X_train_processed))

print("\nTest Validation:")
print(check_invalid_values(X_test_processed))

# Display first 5 rows and 10 columns
print("\nProcessed Data Sample:")
sample = X_train_processed[:5, :10]

if sparse.issparse(sample):
    sample = sample.toarray()

display(pd.DataFrame(sample))


Train Shape: (1296675, 2159)
Test Shape: (555719, 2159)

Same Number of Features: True

Train Validation:
{'Missing Values': np.int64(0), 'Infinite Values': np.int64(0)}

Test Validation:
{'Missing Values': np.int64(0), 'Infinite Values': np.int64(0)}

Processed Data Sample:


,0,1,2,3,4,5,6,7,8,9
0,-0.407826,-0.749136,-0.484420,0.657620,-0.282589,-0.494354,0.593864,-1.878145,-0.941975,-0.890778
1,0.230039,1.872567,2.039120,-2.033870,-0.293670,2.078699,-2.030341,-1.878145,-0.941975,-0.315967
2,0.934149,1.281042,0.717754,-1.601537,-0.280406,0.902849,-1.592323,-1.878145,-0.941975,0.603731
3,-0.158132,0.402753,1.515617,-1.590766,-0.287742,1.662886,-1.621848,-1.878145,-0.941975,0.316326
4,-0.177094,-0.906090,-0.023035,0.782279,-0.293835,0.026941,0.841909,-1.878145,-0.941975,-0.775816



### 4.2 Save Prepared Data and Preprocessing Pipeline

Save the processed training and testing datasets, target variables, and fitted preprocessing pipeline for future machine learning tasks.

- Save processed features in sparse format to reduce memory usage.
- Save target variables separately.
- Save the fitted preprocessing pipeline for consistent transformations.
- Save the final feature names for future reference.

No machine learning model is selected or trained.


In [25]:

import os
import joblib
from scipy import sparse

# Create output folder
os.makedirs("task2_output", exist_ok=True)

# Save processed features
sparse.save_npz(
    "task2_output/X_train_processed.npz",
    sparse.csr_matrix(X_train_processed)
)

sparse.save_npz(
    "task2_output/X_test_processed.npz",
    sparse.csr_matrix(X_test_processed)
)

# Save target variables
y_train.to_csv("task2_output/y_train.csv", index=False)
y_test.to_csv("task2_output/y_test.csv", index=False)

# Save fitted preprocessor
joblib.dump(
    preprocessor,
    "task2_output/preprocessor.joblib"
)

# Save feature names
feature_names = preprocessor.get_feature_names_out()

pd.DataFrame({
    "Feature Name": feature_names
}).to_csv(
    "task2_output/feature_names.csv",
    index=False
)

print("All Task 2 files saved successfully!")


All Task 2 files saved successfully!



## Conclusion

Task 2 successfully prepared the Credit Card Fraud Detection dataset for future machine learning tasks.

- Created four new features through feature engineering.
- Removed unnecessary identifiers and redundant columns.
- Preserved the original training and testing datasets.
- Applied missing value imputation, categorical encoding, and numerical scaling.
- Prevented data leakage by fitting preprocessing only on training data.
- Obtained 2,159 processed features for both datasets.
- Verified that no missing or infinite values remain.
- Saved the processed datasets and fitted preprocessing pipeline.

**The dataset is now ready for model selection and training in Task 3. No machine learning model was trained in Task 2.**
